# Chapter 6 - Attention Heatmaps (live)

Companion notebook for [Chapter 6](../part-2-deep-learning/06-transformer-from-scratch.md). Attention is just `softmax(QK^T / sqrt(d)) V` - and the `softmax(QK^T)` matrix is **interpretable**: it shows which tokens each token looks at. Here we *see* those weights as heatmaps, add a causal mask, and compare multiple heads.

> Requires only `numpy` and `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    w = softmax(scores, axis=-1)
    return w @ V, w

## 1. Token-identity attention

Give each word a random embedding (the two `the`s share one). With `Q = K = embeddings`, each token attends most to tokens whose embedding aligns with its own - so repeated words attend to each other. The heatmap makes this visible.

In [ ]:
tokens = ['the', 'cat', 'sat', 'on', 'the', 'mat']
rng = np.random.default_rng(0)
vocab = {w: rng.standard_normal(16) for w in set(tokens)}
E = np.stack([vocab[w] for w in tokens]) * 2.0        # scale -> sharper softmax

_, w = attention(E, E, E)

plt.figure(figsize=(5, 4.2))
plt.imshow(w, cmap='viridis')
plt.xticks(range(len(tokens)), tokens); plt.yticks(range(len(tokens)), tokens)
plt.xlabel('attends to (key)'); plt.ylabel('query token')
plt.title('Attention weights - note the two "the" tokens')
plt.colorbar(fraction=0.046); plt.tight_layout(); plt.show()
print('rows sum to 1:', np.allclose(w.sum(-1), 1.0))

## 2. The causal mask turns attention into a decoder

Setting future positions to `-inf` *before* softmax means token `i` can only see tokens `<= i` - the weight matrix becomes lower-triangular. This single change is what makes autoregressive generation possible.

In [ ]:
T = len(tokens)
causal = np.tril(np.ones((T, T), dtype=bool))
_, wc = attention(E, E, E, mask=causal)

plt.figure(figsize=(5, 4.2))
plt.imshow(wc, cmap='viridis')
plt.xticks(range(T), tokens); plt.yticks(range(T), tokens)
plt.xlabel('attends to (key)'); plt.ylabel('query token')
plt.title('Causal mask -> lower-triangular (no peeking ahead)')
plt.colorbar(fraction=0.046); plt.tight_layout(); plt.show()
print('future weights are zero:', np.allclose(np.triu(wc, k=1), 0.0))

## 3. Multiple heads learn different patterns

Each head projects Q/K into its own subspace, so different heads attend differently. Here we use random projections to *illustrate* that diversity (a trained model specializes heads to syntax, coreference, induction, etc.).

In [ ]:
d_model, n_heads = 16, 4
d_head = d_model // n_heads
X = rng.standard_normal((T, d_model))

fig, axes = plt.subplots(1, n_heads, figsize=(13, 3.2))
for h in range(n_heads):
    Wq = rng.standard_normal((d_model, d_head))
    Wk = rng.standard_normal((d_model, d_head))
    _, wh = attention(X @ Wq, X @ Wk, X, mask=causal)
    axes[h].imshow(wh, cmap='viridis')
    axes[h].set_title(f'head {h}')
    axes[h].set_xticks(range(T)); axes[h].set_xticklabels(tokens, rotation=90, fontsize=7)
    axes[h].set_yticks(range(T)); axes[h].set_yticklabels(tokens, fontsize=7)
plt.suptitle('Different heads, different attention patterns (random init)')
plt.tight_layout(); plt.show()

## Takeaway

The attention matrix is a window into the model: each row is a probability distribution over which tokens that position pulls information from. Causal masking enforces left-to-right flow; multiple heads capture multiple relationship types in one layer.

**Try it:** swap in your own sentence with repeated words and watch them light up; or remove the `sqrt(d_k)` scaling and see the softmax saturate (one near-1 weight per row).

[Back to Chapter 6](../part-2-deep-learning/06-transformer-from-scratch.md) | [Solutions](../solutions/06-transformer-solutions.md)